**The Afternoon Coffee Consultation**

Alex is currently trapped in the "Founder's Dilemma" where he is working harder but not necessarily smarter. Our first step is to set up our optimization engine using the PuLP library and define the base constraints that currently govern his weekdays and his chaotic Saturdays.

In [2]:
# Step 1: Install and Import PuLP
!pip install pulp -q
from pulp import *

# Step 2: Define the Problem

# Scenario A: Weekday (2 People, 360 min capacity)
# Scenario B: Saturday (3 People, 540 min capacity)

# Introduce Constraints based on Alex's observations
def solve_afternoon_coffee_pro(labor_capacity, scenario_name):
    model = LpProblem(f"Consultation_{scenario_name}", LpMaximize)

    # Decision Variables
    d = LpVariable('Drip', lowBound=0, cat='Integer')
    s = LpVariable('Specialty', lowBound=0, cat='Integer')
    bc = LpVariable('Bacon_Cheese', lowBound=0, cat='Integer')
    ebc = LpVariable('Egg_Bacon_Cheese', lowBound=0, cat='Integer')
    p = LpVariable('Pastry', lowBound=0, cat='Integer')

    # Objective
    model += 3.30*d + 3.65*s + 4.75*bc + 5.50*ebc + 3.00*p

    # --- THE CONSULTING CONSTRAINTS ---

    # 1. Labor Constraint
    model += 1.0*d + 4.0*s + 3.0*bc + 4.0*ebc + 0.5*p <= labor_capacity

    # 2. Market Demand (The "Ceiling")
    # On Saturday, demand is 3x higher [cite: 10]
    mult = 3 if scenario_name == "Saturday" else 1
    model += d <= 40 * mult
    model += s <= 30 * mult
    model += (bc + ebc) <= 20 * mult  # Sandwich limit
    model += p <= 25 * mult           # Pastry limit

    # 3. Customer Loyalty (The "Floor")
    # Alex can't fire his regulars' favorite items
    model += d >= 10
    model += s >= 5

    model.solve(PULP_CBC_CMD(msg=0))

    print(f"--- Results for {scenario_name} ---")
    for v in model.variables():
        print(f"{v.name}: {v.varValue}")
    print(f"Total Profit: ${value(model.objective):.2f}")

    # Return the "Shadow Price" (Dual Value) of Labor
    # This tells us: "If Alex adds 1 more minute of labor, how much does profit grow?"
    return model

# Run the refined model
weekday_model = solve_afternoon_coffee_pro(360, "Weekday")
saturday_model = solve_afternoon_coffee_pro(540, "Saturday")

--- Results for Weekday ---
Bacon_Cheese: 0.0
Drip: 40.0
Egg_Bacon_Cheese: 20.0
Pastry: 25.0
Specialty: 30.0
Total Profit: $426.50
--- Results for Saturday ---
Bacon_Cheese: 58.0
Drip: 120.0
Egg_Bacon_Cheese: 2.0
Pastry: 75.0
Specialty: 50.0
Total Profit: $1090.00


**Analyzing the Surge**

The output shows us that Saturday is a completely different beast compared to a Tuesday. While Alex makes a respectable profit on weekdays, the Saturday "Run Club" surge pushes his labor to the absolute limit. We noticed that on Saturdays, the model is forced to pick faster, lower-margin items just to keep the line moving.

In [3]:


# --- DATA INITIALIZATION ---
items = ["Drip", "Specialty", "Bacon_Cheese", "Egg_Bacon_Cheese", "Pastry"]
profit = {"Drip": 3.30, "Specialty": 3.65, "Bacon_Cheese": 4.75, "Egg_Bacon_Cheese": 5.50, "Pastry": 3.00}
labor_time = {"Drip": 1.0, "Specialty": 4.0, "Bacon_Cheese": 3.0, "Egg_Bacon_Cheese": 4.0, "Pastry": 0.5}

def solve_saturday_surge(labor_capacity, scenario_name):
    # Initialize Problem
    model = LpProblem(f"Saturday_{scenario_name}", LpMaximize)

    # Decision Variables
    vars = LpVariable.dicts("Qty", items, lowBound=0, cat='Integer')

    # Objective Function: Maximize Daily Profit
    model += lpSum([profit[i] * vars[i] for i in items])

    # --- CONSTRAINTS ---

    # 1. Labor Constraint (Total Minutes Available)
    model += lpSum([labor_time[i] * vars[i] for i in items]) <= labor_capacity

    # 2. Demand Planning (Saturday "Run Club" Surge)
    model += vars["Drip"] <= 120
    model += vars["Specialty"] <= 90
    model += vars["Bacon_Cheese"] + vars["Egg_Bacon_Cheese"] <= 60 # Total sandwich cap
    model += vars["Pastry"] <= 75

    # 3. Customer Floor (Keeping the Regulars Happy)
    model += vars["Drip"] >= 30
    model += vars["Specialty"] >= 15

    # Solve quietly
    model.solve(PULP_CBC_CMD(msg=0))

    # Organize results
    res_qty = {v.name.replace("Qty_",""): v.varValue for v in model.variables()}
    total_p = value(model.objective)

    return res_qty, total_p

# --- EXECUTION ---
# Current Saturday: 3 people (Alex + 2) for 3-hour peak (180 mins each) = 540 total mins [cite: 12]
curr_qty, curr_profit = solve_saturday_surge(540, "3_People")

# Proposed Saturday: 4 people (Alex + 3) for 3-hour peak = 720 total mins [cite: 12]
prop_qty, prop_profit = solve_saturday_surge(720, "4_People")

# --- CONSULTATION SUMMARY ---
print(f"{'='*40}")
print(f" AFTERNOON COFFEE: SATURDAY SURGE ANALYSIS")
print(f"{'='*40}")
print(f"SCENARIO 1: 3 EMPLOYEES (540 MINS)")
print(f"Total Profit: ${curr_profit:,.2f}")
print(f"Product Mix: {curr_qty}")
print("-" * 40)
print(f"SCENARIO 2: 4 EMPLOYEES (720 MINS)")
print(f"Total Profit: ${prop_profit:,.2f}")
print(f"Product Mix: {prop_qty}")
print("-" * 40)
print(f"NET GAIN FROM ADDING 4TH PERSON: ${prop_profit - curr_profit:,.2f}")
print(f"LABOR COST (3 hrs @ $15/hr): $45.00")
print(f"FINAL NET PROFIT INCREASE: ${(prop_profit - curr_profit) - 45:,.2f}")
print(f"{'='*40}")

 AFTERNOON COFFEE: SATURDAY SURGE ANALYSIS
SCENARIO 1: 3 EMPLOYEES (540 MINS)
Total Profit: $1,090.00
Product Mix: {'Bacon_Cheese': 58.0, 'Drip': 120.0, 'Egg_Bacon_Cheese': 2.0, 'Pastry': 75.0, 'Specialty': 50.0}
----------------------------------------
SCENARIO 2: 4 EMPLOYEES (720 MINS)
Total Profit: $1,251.00
Product Mix: {'Bacon_Cheese': 38.0, 'Drip': 120.0, 'Egg_Bacon_Cheese': 22.0, 'Pastry': 75.0, 'Specialty': 90.0}
----------------------------------------
NET GAIN FROM ADDING 4TH PERSON: $161.00
LABOR COST (3 hrs @ $15/hr): $45.00
FINAL NET PROFIT INCREASE: $116.00


**The Verdict on the Fourth Person**

By running the math for a fourth employee, we found that the investment pays for itself instantly. Adding that extra set of hands doesn't just make things less stressful. It actually allows Alex to flip his production from low-margin sandwiches to the high-margin "Egg, Bacon, and Cheese" options that his customers are currently missing out on.

In [4]:


# --- DATA INITIALIZATION ---
#  alex (owner) + current employees define the labor pool
items = ["Drip", "Specialty", "Bacon_Cheese", "Egg_Bacon_Cheese", "Pastry"]
profit = {"Drip": 3.30, "Specialty": 3.65, "Bacon_Cheese": 4.75, "Egg_Bacon_Cheese": 5.50, "Pastry": 3.00}
labor_time = {"Drip": 1.0, "Specialty": 4.0, "Bacon_Cheese": 3.0, "Egg_Bacon_Cheese": 4.0, "Pastry": 0.5}

def solve_scenario(labor_capacity, demand_multiplier, scenario_name):
    # Initialize Problem
    model = LpProblem(f"Afternoon_Coffee_{scenario_name}", LpMaximize)

    # Decision Variables
    vars = LpVariable.dicts("Qty", items, lowBound=0, cat='Integer')

    # Objective Function: Maximize Profit
    model += lpSum([profit[i] * vars[i] for i in items])

    # --- CONSTRAINTS ---

    # 1. Labor Constraint
    model += lpSum([labor_time[i] * vars[i] for i in items]) <= labor_capacity

    # 2. Demand Planning [cite: 10, 11]
    # weekday base demand
    model += vars["Drip"] <= 40 * demand_multiplier
    model += vars["Specialty"] <= 30 * demand_multiplier
    model += vars["Bacon_Cheese"] + vars["Egg_Bacon_Cheese"] <= 20 * demand_multiplier
    model += vars["Pastry"] <= 25 * demand_multiplier

    # 3. Customer Loyalty (Floor)
    model += vars["Drip"] >= 10 * demand_multiplier
    model += vars["Specialty"] >= 5 * demand_multiplier

    # Solve
    model.solve(PULP_CBC_CMD(msg=0))

    res_qty = {v.name.replace("Qty_",""): v.varValue for v in model.variables()}
    total_p = value(model.objective)

    return res_qty, total_p

# --- EXECUTION ---

# WEEKDAY 1st Shift (9 AM - 12 PM - 3 hours)
# Current: 2 people (Alex + 1) = 360 mins
wk_curr_qty, wk_curr_profit = solve_scenario(360, 1, "Weekday_2_Staff")
# Proposed: 3 people (Alex + 2) = 540 mins
wk_prop_qty, wk_prop_profit = solve_scenario(540, 1, "Weekday_3_Staff")

# SUNDAY Shift (8 AM - 2 PM - 6 hours)
# Assume demand is slightly higher than weekday (1.5x)
# Current: 2 people = 720 mins
sun_curr_qty, sun_curr_profit = solve_scenario(720, 1.5, "Sunday_2_Staff")
# Proposed: 3 people = 1080 mins
sun_prop_qty, sun_prop_profit = solve_scenario(1080, 1.5, "Sunday_3_Staff")

# --- RESULTS SUMMARY ---
def print_results(day, current_p, prop_p, labor_hrs, labor_cost_per_hr=15):
    net_gain = prop_p - current_p
    cost = labor_hrs * labor_cost_per_hr
    roi = net_gain - cost
    print(f"ANALYSIS FOR: {day}")
    print(f"Profit with Current Staff: ${current_p:,.2f}")
    print(f"Profit with Extra Staff:   ${prop_p:,.2f}")
    print(f"Gross Gain:                ${net_gain:,.2f}")
    print(f"Cost of Extra Labor:       ${cost:,.2f}")
    print(f"NET PROFIT IMPACT:         ${roi:,.2f}")
    if roi > 0:
        print("RECOMMENDATION: HIRE (Profitable)")
    else:
        print("RECOMMENDATION: DO NOT HIRE (Not profitable based on demand caps)")
    print("-" * 40)

print("="*40)
print(" AFTERNOON COFFEE: LABOR EXPANSION REPORT")
print("="*40)
# Weekday shift is 3 hours for the extra person
print_results("Weekday 1st Shift (3 hrs)", wk_curr_profit, wk_prop_profit, 3)
# Sunday shift is 6 hours
print_results("Sunday Full Shift (6 hrs)", sun_curr_profit, sun_prop_profit, 6)

 AFTERNOON COFFEE: LABOR EXPANSION REPORT
ANALYSIS FOR: Weekday 1st Shift (3 hrs)
Profit with Current Staff: $426.50
Profit with Extra Staff:   $426.50
Gross Gain:                $0.00
Cost of Extra Labor:       $45.00
NET PROFIT IMPACT:         $-45.00
RECOMMENDATION: DO NOT HIRE (Not profitable based on demand caps)
----------------------------------------
ANALYSIS FOR: Sunday Full Shift (6 hrs)
Profit with Current Staff: $638.25
Profit with Extra Staff:   $638.25
Gross Gain:                $0.00
Cost of Extra Labor:       $90.00
NET PROFIT IMPACT:         $-90.00
RECOMMENDATION: DO NOT HIRE (Not profitable based on demand caps)
----------------------------------------


**Finding the Inventory "Sweet Spot"**

One of Alex's biggest headaches is throwing away food at 3 pm. This final block of code finds the exact preparation quantities to keep regulars happy without creating waste. We also added a 20% "Breather Buffer" to ensure Alex has time to keep the shop clean and stocked, solving the "Overwhelming" feeling he described.

In [9]:


def solve_alex_sweet_spot(labor_capacity=360):
    model = pulp.LpProblem("Alex_Inventory_Optimization", pulp.LpMaximize)

    items = ["Drip", "Specialty", "Bacon_Cheese", "Egg_Bacon_Cheese", "Pastry"]
    profit = {"Drip": 3.30, "Specialty": 3.65, "Bacon_Cheese": 4.75, "Egg_Bacon_Cheese": 5.50, "Pastry": 3.00}
    labor_time = {"Drip": 1.0, "Specialty": 4.0, "Bacon_Cheese": 3.0, "Egg_Bacon_Cheese": 4.0, "Pastry": 0.5}

    vars = pulp.LpVariable.dicts("Qty", items, lowBound=0, cat='Integer')

    # Objective Function
    model += pulp.lpSum([profit[i] * vars[i] for i in items])

    # --- THE CONSTRAINTS ---

    # 1. Labor with 20% "Breather Buffer" (Alex's request for operations/cleaning)
    # 360 mins * 0.80 = 288 usable minutes for orders
    model += pulp.lpSum([labor_time[i] * vars[i] for i in items]) <= (labor_capacity * 0.80)

    # 2. Market Demand Ceiling (Prevents overproduction and wastage)
    model += vars["Drip"] <= 50             # Alex's actual weekday max
    model += vars["Specialty"] <= 30        # Alex's actual weekday max
    model += vars["Bacon_Cheese"] <= 12     # Perishability Cap
    model += vars["Egg_Bacon_Cheese"] <= 8  # Perishability Cap
    model += vars["Pastry"] <= 20           # Perishability Cap

    # 3. Loyalty Floor (Regulars' Favorites)
    model += vars["Specialty"] >= 10
    model += vars["Pastry"] >= 5

    model.solve(pulp.PULP_CBC_CMD(msg=0))

    # Print Results
    print(f"--- REFINED WEEKDAY SWEET SPOT ---")
    for i in items:
        print(f"{i}: {vars[i].varValue}")

    print(f"Total Profit: ${pulp.value(model.objective):.2f}")
    print(f"Labor Used: {sum(labor_time[i] * vars[i].varValue for i in items)} / {labor_capacity} mins")

# Execute the search for the sweet spot
solve_alex_sweet_spot(360)

--- REFINED WEEKDAY SWEET SPOT ---
Drip: 50.0
Specialty: 30.0
Bacon_Cheese: 12.0
Egg_Bacon_Cheese: 8.0
Pastry: 20.0
Total Profit: $435.50
Labor Used: 248.0 / 360 mins


Through this methodology, we have moved Alex from "guessing" to "knowing." He now has a data-backed plan to handle the Saturday surge, a way to minimize his weekday waste, and proof that he can take a breather without losing his profit.